# Model
BERT-style masked transformer encoder for DESI spectra.
Sequence: [CLS] [Z_MASK] patch_1 ... patch_253 (255 tokens total).

In [ ]:
import torch
import torch.nn as nn
import sys, os
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
import config
%run tokenizer.ipynb
%run masking.ipynb

In [ ]:
class SpectralTransformer(nn.Module):
    """BERT-style masked transformer encoder for DESI astrophysical spectra.

    The [Z_MASK] token is always masked; the model infers redshift from spectral context.
    Masked spectral patches are replaced with a learned mask embedding before encoding.

    Args:
        d_model (int): Token embedding dimension.
        n_heads (int): Number of self-attention heads.
        n_layers (int): Number of transformer encoder layers.
        d_ffn (int): Feed-forward network hidden dimension.
        patch_size (int): Number of flux pixels per patch.
        n_patches (int): Number of spectral patches.
        max_seq_len (int): Total sequence length including special tokens.
    """

    def __init__(
        self,
        d_model=config.D_MODEL,
        n_heads=config.N_HEADS,
        n_layers=config.N_LAYERS,
        d_ffn=config.D_FFN,
        patch_size=config.PATCH_SIZE,
        n_patches=config.N_PATCHES,
        max_seq_len=config.MAX_SEQ_LEN,
    ):
        super().__init__()
        self.n_patches = n_patches
        self.patch_size = patch_size

        self.cls_token     = nn.Parameter(torch.zeros(1, 1, d_model))
        self.z_mask_token  = nn.Parameter(torch.zeros(1, 1, d_model))
        self.mask_embedding = nn.Parameter(torch.zeros(1, 1, d_model))

        self.patch_embed = PatchEmbedding(patch_size, d_model)

        self.register_buffer(
            'pos_enc',
            sinusoidal_positional_encoding(max_seq_len, d_model).unsqueeze(0),
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ffn,
            dropout=0.1,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.reconstruction_head = nn.Linear(d_model, patch_size)
        self.redshift_head = nn.Sequential(
            nn.Linear(d_model, 192),
            nn.GELU(),
            nn.Linear(192, 64),
            nn.GELU(),
            nn.Linear(64, 1),
        )

        self._init_weights()

    def _init_weights(self):
        """Initialize special token parameters with truncated normal distribution."""
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.z_mask_token, std=0.02)
        nn.init.trunc_normal_(self.mask_embedding, std=0.02)

    def forward(self, flux, mask):
        """Encode a batch of spectra and produce reconstruction and redshift predictions.

        Args:
            flux (torch.Tensor): Padded flux arrays of shape (B, PADDED_LEN).
            mask (torch.Tensor): Boolean mask of shape (B, N_PATCHES); True = masked patch.

        Returns:
            tuple:
                recon_patches (torch.Tensor): Reconstructed flux at masked positions,
                    shape (B, n_masked, PATCH_SIZE).
                z_pred (torch.Tensor): Predicted redshift values, shape (B, 1).
        """
        B = flux.shape[0]

        patch_tokens = self.patch_embed(flux)
        patch_tokens = apply_mask(patch_tokens, mask, self.mask_embedding)

        cls   = self.cls_token.expand(B, -1, -1)
        z_tok = self.z_mask_token.expand(B, -1, -1)
        tokens = torch.cat([cls, z_tok, patch_tokens], dim=1)
        tokens = tokens + self.pos_enc

        encoded = self.encoder(tokens)

        z_pred = self.redshift_head(encoded[:, 1, :])

        patch_encoded = encoded[:, 2:, :]
        n_masked = mask.sum(dim=1)[0].item()
        masked_encoded = patch_encoded[mask].reshape(B, int(n_masked), -1)
        recon_patches = self.reconstruction_head(masked_encoded)

        return recon_patches, z_pred

In [ ]:
if __name__ == '__main__':
    model = SpectralTransformer()
    total = sum(p.numel() for p in model.parameters())
    print(f'Total parameters: {total:,}')
    flux = torch.randn(2, config.PADDED_LEN)
    mask = random_mask(2, config.N_PATCHES)
    recon, z = model(flux, mask)
    print(f'recon: {recon.shape}  z_pred: {z.shape}')